# 02 — 2D Cantilever Beam

**Curriculum slot:** Tier 1, slot 02.  
**Prerequisite:** 01 — Hello Plate.  
**Strategy used for results:** `spec.emit_default(...)` — on pip openseespy, live classic OpenSees recorders.

## Problem

A prismatic cantilever of length $L$ is fully fixed at $x = 0$ and
loaded at its free tip by a vertical point force $P$ (downward).

```
                                P↓
  (base)●──────────────────────●(tip)        +z
         ◀─────  L = 3 m  ─────▶              │
                                              └── +x
```

Linear-elastic Euler-Bernoulli theory gives the closed-form tip
deflection

$$
\delta_{\text{tip}} \;=\; -\,\dfrac{P\,L^{3}}{3\,E\,I}
$$

and the deflected shape

$$
w(x) \;=\; -\,\dfrac{P\,x^{2}}{6\,E\,I}\,(3L - x).
$$

## Why this notebook is interesting

* It looks inside the default route from 01. On pip openseespy,
  `spec.emit_default(...)` pushes classic `ops.recorder("Node", ...)`
  calls into the live `ops` domain, the output `.out` files land on
  disk, and `run.results()` transcodes them with
  `Results.from_recorders(...)`. On the Ladruno fork the same cell
  writes one `.ladruno` file instead.
* Visualisation: the deflected shape vs the closed-form
  Euler-Bernoulli cubic, all driven from `Results.nodes.get(...)`.

## 1. Imports and parameters

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openseespy.opensees as ops

from apeGmsh import apeGmsh
from apeGmsh.results.spec import ResolvedRecorderRecord, ResolvedRecorderSpec

# Geometry / load (consistent units: m, Pa, N)
L = 3.0
P = 10_000.0     # tip force magnitude (applied as -P in +z)

# Elastic material / cross-section (3D beam-column needs all of these)
E  = 2.1e11
nu = 0.3
A  = 1.0e-3
Iy = 1.0e-5
Iz = 1.0e-5     # bending about y-axis -> vertical deflection in +z
J  = 2.0e-5
G  = E / (2.0 * (1.0 + nu))

# Mesh density
LC = L / 10.0

## 2. Geometry

A single line — base point at the origin, tip point at $x = L$.

In [ ]:
from apeGmsh import workdir
g = apeGmsh(model_name="02_cantilever_beam_2D", save_to=workdir() / "02_cantilever_beam_2D.h5", verbose=False)
g.begin()

p_base = g.model.geometry.add_point(0.0, 0.0, 0.0, lc=LC)
p_tip  = g.model.geometry.add_point(L,   0.0, 0.0, lc=LC)
line   = g.model.geometry.add_line(p_base, p_tip)

g.model.sync()

## 3. Physical groups

Three names — `base` for the fixed end, `tip` for the loaded end,
`beam` for the line that becomes the elements.

In [ ]:
g.physical.add(0, [p_base], name="base")
g.physical.add(0, [p_tip],  name="tip")
g.physical.add(1, [line],   name="beam")

## 4. Mesh

In [ ]:
g.mesh.generation.generate(1)
fem = g.mesh.queries.get_fem_data()
print(f"mesh built: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

## 5. Build the OpenSees model — vanilla openseespy

The 'beam' is `elasticBeamColumn` in 3D (`-ndm 3 -ndf 6`) with a
linear geometric transformation. Base fully fixed; tip carries
$-P$ in $+z$.

In [ ]:
ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 6)

# nodes
for nid, xyz in fem.nodes.select():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

# geometric transformation (local-y aligned with global +y)
TRANSF_TAG = 1
ops.geomTransf("Linear", TRANSF_TAG, 0.0, 1.0, 0.0)

# elastic-beam-column elements
for group in fem.elements.select(target="beam").result():
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "elasticBeamColumn", int(eid),
            int(conn[0]), int(conn[1]),
            A, E, G, J, Iy, Iz, TRANSF_TAG,
        )

# fully fix the base point
for n in fem.nodes.select(target="base").ids:
    ops.fix(int(n), 1, 1, 1, 1, 1, 1)

# downward point load at the tip
ops.timeSeries("Constant", 1)
ops.pattern("Plain", 1, 1)
for n in fem.nodes.select(target="tip").ids:
    ops.load(int(n), 0.0, 0.0, -P, 0.0, 0.0, 0.0)

## 6. Declare recorders

Two named records:

* displacement on **all nodes** — full deflection field for the plot.
* reaction on the `"base"` group — both forces and moments: the six
  components `reaction_force_x..z` and `reaction_moment_x..z` of an
  `ndf=6` node.

Modal records are intentionally omitted — no default route records
modes; notebook 17 covers eigenmodes.

In [ ]:
spec = ResolvedRecorderSpec(
    fem_snapshot_id=fem.snapshot_id,
    records=(
        ResolvedRecorderRecord(
            category="nodes", name="u_all",
            components=("displacement_x", "displacement_y", "displacement_z"),
            dt=None, n_steps=None, node_ids=fem.nodes.ids,
        ),
        ResolvedRecorderRecord(
            category="nodes", name="R_base",
            components=("reaction_force_x", "reaction_force_y", "reaction_force_z", "reaction_moment_x", "reaction_moment_y", "reaction_moment_z"),
            dt=None, n_steps=None, node_ids=fem.nodes.select(pg="base").ids,
        ),
    ),
)
print(spec)

## 7. Run the analysis with `spec.emit_default(...)`

What the route writes depends on the OpenSees build (`run.route` says
which ran):

| Build | Who writes the file | Output | Read back |
|---|---|---|---|
| pip openseespy | OpenSees `Node` recorders | one `.out` per record in `outputs/run_out/` | `Results.from_recorders(...)` |
| stock with MPCO | the MPCO recorder | `outputs/run.mpco` | `Results.from_mpco(...)` |
| Ladruno fork | the fork's own recorder | `outputs/run.ladruno` | `Results.from_ladruno(...)` |

No per-step hook is needed: the recorders fire on every converged
`analyze`, and leaving the `with` block closes them, which flushes the
files. Every route yields the same `Results` API on read.

In [ ]:
from apeGmsh import workdir
OUT = workdir()
model_h5 = OUT / "model.h5"
fem.to_h5(str(model_h5))              # the model archive the read pairs with
# On the fork, run.results() warns once that run.ladruno is read unstamped:
# a fem.to_h5 archive carries no model_hash to pair with yet (#2233).

ops.system("BandGeneral")
ops.numberer("Plain")
ops.constraints("Plain")
ops.test("NormUnbalance", 1e-10, 10)
ops.algorithm("Linear")
ops.integrator("LoadControl", 1.0)
ops.analysis("Static")

with spec.emit_default(OUT / "run", model_h5=model_h5, ops=ops,
                       overwrite=True) as run:
    status = ops.analyze(1)
    assert status == 0, f"ops.analyze returned {status}"

print(f"route: {run.route} -> {run.path}")

## 8. Read results back — `run.results()`

`run.results()` opens what the route wrote. On pip it transcodes the
`.out` files into a native HDF5 (cached on disk by mtime + spec hash),
and the resulting `Results` object exposes the same composite API as
in notebook 01.

In [ ]:
results = run.results(cache_root=OUT / "cache")
print(results.inspect.summary())

In [ ]:
# Tip vertical deflection (verification target)
tip_id = int(next(iter(fem.nodes.select(target="tip").ids)))
u_tip = results.nodes.get(
    component="displacement_z", ids=[tip_id], time=-1,
)
delta_fem    = float(u_tip.values[0, 0])
delta_closed = -P * L**3 / (3.0 * E * Iz)
print(f"tip deflection (FEM)        :  {delta_fem:+.6e}  m")
print(f"−P L^3 / (3 E I)            :  {delta_closed:+.6e}  m   (closed form)")
print(f"relative error              :  {abs(delta_fem - delta_closed) / abs(delta_closed):.2e}")

In [ ]:
# Base reaction — equilibrium check
R_z = results.nodes.get(
    component="reaction_force_z", pg="base", time=-1,
).values.sum()
M_y = results.nodes.get(
    component="reaction_moment_y", pg="base", time=-1,
).values.sum()
print(f"Σ R_z at base               :  {R_z:+.6e}  N    (equilibrium: +P = {P:+.6e})")
print(f"Σ M_y at base               :  {M_y:+.6e}  N·m  (closed form: −P L = {-P * L:+.6e})")

## 9. Plot the deflected shape vs Euler-Bernoulli

Pull `displacement_z` at every node, pair each value with the node's
$x$-coordinate via the bound FEM, and overlay the closed-form curve.

Linear elastic, prismatic section, point load at the tip — the FEM
cubic should track the analytical cubic to round-off.

In [ ]:
u_all = results.nodes.get(component="displacement_z", time=-1)
x_lookup = dict(zip(
    np.asarray(fem.nodes.ids, dtype=np.int64).tolist(),
    np.asarray(fem.nodes.coords)[:, 0].tolist(),
))
x_arr = np.array([x_lookup[int(n)] for n in u_all.node_ids])
u_arr = u_all.values[0]
order = np.argsort(x_arr)

x_fine = np.linspace(0, L, 200)
w_closed = -P * x_fine**2 / (6.0 * E * Iz) * (3.0 * L - x_fine)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_fine, w_closed, "-", color="k", lw=1, label="closed-form  −Px²(3L−x)/(6EI)")
ax.plot(x_arr[order], u_arr[order], "o", ms=5, color="C0", label="FEM (emit_default)")
ax.set_xlabel("x  [m]"); ax.set_ylabel("w(x)  [m]")
ax.set_title("Cantilever deflected shape")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

## 10. (Optional) Open the post-solve viewer

The viewer also opens a transcoded-from-recorders `Results` object —
the read API is identical regardless of which strategy produced the file.

In [ ]:
results.viewer(blocking=False)


## What this unlocks

* You've now seen what the default route does on each build: classic
  recorder `.out` files on pip openseespy, an `.mpco` on a stock build
  with MPCO, the fork's `.ladruno` on the Ladruno fork.
* Same declaration vocabulary and the same `Results` read API, whichever
  file the run produced.
* The trade-off is mostly format: the HDF5 routes carry more per file,
  and the classic recorders keep OpenSees recorder semantics in
  cluster-friendly text files.

From here the curriculum turns to multi-element models (notebook 04 —
portal frame), naming patterns (05 — labels and PGs), and beyond.

In [ ]:
g.end()